In [4]:
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.model_selection import PredefinedSplit
from sklearn.utils import resample
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

seed = 1234

In [ ]:
def accuracy_visual(classifier, X_tr, X_te, y_tr, y_te):
    tr = classifier.predict(X_tr)
    te = classifier.predict(X_te)
    as_tr = accuracy_score(y_tr, tr)
    as_te = accuracy_score(y_te, te)

    fig, ax = plt.subplots()
    bars = ax.bar(['Train', 'Train Baseline', 'Test', 'Test Baseline'], 
                  [as_tr*100, max(y_tr.mean(), 1 - y_tr.mean())*100, 
                   as_te*100, max(y_te.mean(), 1 - y_te.mean())*100], color=['steelblue', 'teal'])
    ax.bar_label(bars, fmt='%.1f%%')
    ax.set_ylabel('Accuracy (%)')
    ax.set_ylim(0, 100)
    plt.show()

    cm = confusion_matrix(y_te, te)
    ConfusionMatrixDisplay.from_predictions(y_te, te)


### ____________________________________

In [11]:
DATA_PATHS = ["processed_data\data_15min_2025.parquet"]
df = pd.DataFrame()
for path in DATA_PATHS:
    df = pd.concat([df, pd.read_parquet(path)], ignore_index=True)
print(df)

            T              t       vw       ema9      ema20      o       c  \
0          AA  1735814700000  38.1062  38.110000  38.110000  38.09  38.110   
1          AA  1735815600000  38.3654  38.162000  38.134762  38.37  38.370   
2          AA  1735820100000  38.2500  38.179600  38.145737  38.25  38.250   
3          AA  1735821900000  38.2406  38.191680  38.154714  38.24  38.240   
4          AA  1735823700000  38.2297  38.201344  38.162837  38.23  38.240   
...       ...            ...      ...        ...        ...    ...     ...   
33589194  ZZZ  1756128600000  29.9674  29.944562  29.972226  30.09  29.865   
33589195  ZZZ  1756130400000  29.9400  29.943649  29.969157  29.94  29.940   
33589196  ZZZ  1756217700000  29.7500  29.904920  29.948285  29.75  29.750   
33589197  ZZZ  1756474200000  30.1600  29.955936  29.968448  30.16  30.160   
33589198  ZZZ  1756486800000  29.7023  29.904748  29.942882  29.70  29.700   

              h       l   n        rv        gp   otc  y  
0   

In [12]:
X = df.drop(columns=["y"])
y = df["y"]

print(X)
print(y)

            T              t       vw       ema9      ema20      o       c  \
0          AA  1735814700000  38.1062  38.110000  38.110000  38.09  38.110   
1          AA  1735815600000  38.3654  38.162000  38.134762  38.37  38.370   
2          AA  1735820100000  38.2500  38.179600  38.145737  38.25  38.250   
3          AA  1735821900000  38.2406  38.191680  38.154714  38.24  38.240   
4          AA  1735823700000  38.2297  38.201344  38.162837  38.23  38.240   
...       ...            ...      ...        ...        ...    ...     ...   
33589194  ZZZ  1756128600000  29.9674  29.944562  29.972226  30.09  29.865   
33589195  ZZZ  1756130400000  29.9400  29.943649  29.969157  29.94  29.940   
33589196  ZZZ  1756217700000  29.7500  29.904920  29.948285  29.75  29.750   
33589197  ZZZ  1756474200000  30.1600  29.955936  29.968448  30.16  30.160   
33589198  ZZZ  1756486800000  29.7023  29.904748  29.942882  29.70  29.700   

              h       l   n        rv        gp   otc  
0      

In [17]:
min_t = df["t"].min()
max_t = df["t"].max()
print(min_t, max_t)

steps = (max_t-min_t)/900000
print(steps)

1735808400000 1767228300000
34911.0


In [ ]:
# Train-Test-Validation = (60-20-20)


In [20]:
X_tr_val, X_te, y_tr_val, y_te = train_test_split(X, y, test_size=0.2, random_state = seed)
X_tr, X_val, y_tr, y_val = train_test_split(X_tr_val, y_tr_val, test_size=0.25, random_state = seed)

# Train-Test-Validation = (60-20-20)
print(X_tr)

KeyboardInterrupt: 

In [ ]:
X_tr_sample, y_tr_sample = resample(X_tr, y_tr, n_samples=10000, random_state = seed)
X_combined = np.vstack([X_tr_sample, X_val])
y_combined = np.concatenate([y_tr_sample, y_val])
split_index = np.concatenate([-np.ones(len(X_tr_sample)), np.zeros(len(X_val))])
ps = PredefinedSplit(split_index)

# Linear Regression

In [ ]:
lr_random_search = RandomizedSearchCV(
    Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000))
    ]),
    lr_random_search_params,
    n_iter = 30,           # combos
    cv = ps,               # cross validation
    random_state=seed
)

lr_random_search.fit(X_combined, y_combined)
best_params = {k.replace("clf__", ""): v for k, v in lr_random_search.best_params_.items()}
print(lr_random_search.best_params_)

lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(random_state=seed, **best_params))
])
lr.fit(X_tr, y_tr)

accuracy_visual(lr, X_tr, X_te, y_tr, y_te)

# Random Forests

In [ ]:
rfc_random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=seed), 
    rfc_random_search_params,
    n_iter = 30,           # combos
    cv = ps,               # cross validation
    random_state=seed
)

rfc_random_search.fit(X_combined, y_combined)
print(rfc_random_search.best_params_)

rfc = RandomForestClassifier(random_state=seed, **rfc_random_search.best_params_)
rfc.fit(X_tr, y_tr)

accuracy_visual(rfc, X_tr, X_te, y_tr, y_te)

# XGB Classifier

In [ ]:
xgbc_random_search = RandomizedSearchCV(
    Pipeline([
        ("scaler", StandardScaler()),
        ("clf", XGBClassifier())
    ]),
    xgbc_random_search_params,
    n_iter = 30,           # combos
    cv = ps,               # cross validation
    random_state=seed
)

xgbc_random_search.fit(X_combined, y_combined)
best_params = {k.replace("clf__", ""): v for k, v in xgbc_random_search.best_params_.items()}
print(xgbc_random_search.best_params_)

xgbc = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", XGBClassifier(random_state=seed, **best_params))
])
xgbc.fit(X_tr, y_tr)

accuracy_visual(xgbc, X_tr, X_te, y_tr, y_te)